## 面试问题

Look-ahead 试探：不产生真实副作用地预演下一步？

## 回答主线

高风险不可逆动作前应先 dry-run 预演，在状态副本上模拟结果而不碰真实状态，据此决定是否真执行。本 Notebook 用转账 agent：两笔转账合计超过余额，对比无 dry-run（直接转导致透支）与 dry-run（预演发现会透支就拒绝，真实余额不变）。

## 真实案例

账户余额 100，两笔转账 60 和 80（合计 140 超余额）。dry-run 在余额副本上模拟扣款，发现第二笔会透支即拒绝。数据为教学账户，不代表真实支付系统。

In [1]:
account = {"balance": 100}  # 账户初始余额。
transfers = [{"to": "X", "amount": 60}, {"to": "Y", "amount": 80}]  # 两笔待处理转账。

print("初始余额:", account["balance"])  # 展示初始余额。
for t in transfers:  # 逐笔打印转账。
    print("  转账:", t["to"], t["amount"])  # 展示每笔转账。

初始余额: 100
  转账: X 60
  转账: Y 80


## 基线（Baseline）

反面基线：无 dry-run，直接执行转账。第二笔 80 超过剩余余额 40，真实余额被扣成负数（透支），且已是不可逆副作用。

In [2]:
def commit_transfer(acct, amount):  # 真执行转账：直接扣减真实余额。
    acct["balance"] -= amount  # 扣减余额产生真实副作用。
    return acct["balance"]  # 返回扣减后余额。

naive_account = {"balance": 100}  # 无预演的账户。
naive_log = []  # 记录每笔结果。
for t in transfers:  # 逐笔直接执行。
    bal = commit_transfer(naive_account, t["amount"])  # 不预演直接转账。
    naive_log.append(bal)  # 记录扣减后余额。
print("无 dry-run 每笔后余额:", naive_log)  # 展示第二笔导致余额为负透支。
print("无 dry-run 最终余额:", naive_account["balance"])  # 展示真实余额已透支。

无 dry-run 每笔后余额: [40, -40]
无 dry-run 最终余额: -40


## 失败案例与修正

无预演直接透支。修正是 dry-run：先在余额副本上模拟本笔转账，预测为负就拒绝、不产生副作用，预测非负才真正 commit。

In [3]:
def dry_run_transfer(acct, amount):  # 预演转账：在状态副本上模拟不碰真实余额。
    shadow = dict(acct)  # 拷贝一个状态副本。
    shadow["balance"] -= amount  # 在副本上模拟扣款。
    return shadow["balance"]  # 返回预演后的余额而不改真实账户。

In [4]:
guarded_account = {"balance": 100}  # 带预演保护的账户。
guarded_log = []  # 记录每笔决策。
for t in transfers:  # 逐笔先预演再决定。
    predicted = dry_run_transfer(guarded_account, t["amount"])  # 预演本笔转账后的余额。
    if predicted < 0:  # 预演发现会透支。
        guarded_log.append(("rejected", t["amount"], predicted))  # 拒绝执行不产生副作用。
    else:  # 预演通过。
        commit_transfer(guarded_account, t["amount"])  # 才真正执行转账。
        guarded_log.append(("committed", t["amount"], guarded_account["balance"]))  # 记录真实扣减。
print("dry-run 每笔决策:", guarded_log)  # 展示第二笔被预演拦截。
print("dry-run 最终余额:", guarded_account["balance"])  # 展示真实余额未透支。

dry-run 每笔决策: [('committed', 60, 40), ('rejected', 80, -40)]
dry-run 最终余额: 40


In [5]:
rejected_count = sum(1 for r in guarded_log if r[0] == "rejected")  # 统计被预演拦截的笔数。
print("无 dry-run 最终余额:", naive_account["balance"], "(透支)")  # 无预演透支。
print("有 dry-run 最终余额:", guarded_account["balance"], "(未透支)")  # 有预演安全。
print("dry-run 拦截笔数:", rejected_count)  # 展示预演拦截了透支转账。

无 dry-run 最终余额: -40 (透支)
有 dry-run 最终余额: 40 (未透支)
dry-run 拦截笔数: 1


## 结果解读

无 dry-run 第二笔把余额扣成 -40（透支且不可逆）；dry-run 在副本上预演发现第二笔会透支，拒绝执行、真实余额停在 40。要点：预演必须无副作用（在副本上做）、保真度要够、只对高风险不可逆动作预演。

In [6]:
assert naive_account["balance"] == -40  # 无 dry-run 导致真实余额透支。
assert guarded_account["balance"] == 40  # 有 dry-run 保护真实余额未透支。
assert guarded_log[0][0] == "committed"  # 第一笔预演通过并执行。
assert guarded_log[1][0] == "rejected"  # 第二笔预演发现透支被拒绝。
assert rejected_count == 1  # 预演共拦截一笔。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
